# Chromatic VIS case: best split of the 700-1500 nm light between two fused pyramids

WFSW error budget (`aoeb`), science at **550 nm only**, z = 30 deg, 0.9" along the line of sight, leaky controller, 250 us latency, contrast = mean raw contrast in 2-5 lambda/D. Code: `aoeb/bandsplit.py` on top of `aoeb.multiwfs.Study`; every budget is cached in `results/bandsplit_cache` (delete it, or `st.recompute = True`, after a model change).

1. **Anchor**: the budget of the e2e configurations (monochromatic 900 / 1300 nm sensors, as in SPECULA) against the replay means of the final e2e sweep; only ratios of the budget are trusted afterwards.
2. **Band model**: a band [lo, hi] collects the star spectrum over the band (colour m900 - m1300 of the e2e star, magnitude linear in the wavelength, flat QE); products of its photon-weighted mean wavelength (derived from 900 / 1300 nm, A37); chromatic shear with its photon-weighted mean refractivity.
3. **Scan**: boundary lb in 750-1450 nm, sensor A = 700-lb, B = lb-1500; unmodulated (rMod 0) on A or on B, the other rMod 2; both 48x48, 1300 modes, one loop at the rate (<= 2 kHz) minimising the residual. Baselines: single 700-1100 (today's sensor) and single 700-1500 (the whole band on one pyramid).
4. **Weights** of the two measurements of every mode: *min_noise* (inverse variance, as the e2e Kalman filter), *uniform* (one shift for all modes, chosen on the contrast zone), *per_mode* (two-colour estimator: per mode the weight minimising loop noise + aliasing + chromatic shear, which extrapolates towards 550 nm).
5. **Output**: contrast maps relative to today's fusion (unmod 700-1100 + rMod 2 1100-1500, min_noise), best split per magnitude, best splits vs magnitude.

Limits: unmodulated sensor with an SVD reconstructor (e2e: ML); 48x48 for both (the chromatic term matters only at bright magnitudes); a band's chromatic spread inside the band is not modelled (second order); products at band wavelengths are scaled from 900 / 1300 nm.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, warnings, importlib.util
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

E2E_DIR = os.path.abspath('aoeb/e2e')                      # run from SPECULA/config
sys.path.insert(0, E2E_DIR)
_spec = importlib.util.spec_from_file_location('aoeb_e2e_utils', os.path.join(E2E_DIR, 'utils.py'))
e2e = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(e2e)
from aoeb.chromatic import ChromaticModel
from aoeb.psf import PsfEngine
from aoeb.multiwfs import Study
from aoeb.bandsplit import Photometry, SplitStudy
import final_sweep_results as fr                            # e2e replay statistics (results/WFSW/sweep_final_20261108)
warnings.filterwarnings('ignore', message='.*(not calibrated at|not simulated for).*')   # band wavelengths: products scaled (A37)

Y = yaml.safe_load(open('aoeb/e2e/ristretto_main.yml'))
m = e2e.machine()                                           # WFSW products, budget_wfsw.yml
m.max_sensitivity = None                                    # as analyse_errorbudget.ipynb
SEEING, ZENITH, SCI_NM, CONTROLLER = 0.9, Y['main']['zenithAngleInDeg'], 550.0, 'leaky'
LATENCY = Y['kpred']['delay'] * Y['main']['time_step']
AREA = (m.pupil() > 0).sum() * Y['main']['pixel_pitch'] ** 2
COLOUR = Y['source_vis']['magnitude'] - Y['source_nir']['magnitude']
st = Study(m, PsfEngine(m), SEEING, LATENCY, CONTROLLER, AREA, bandw=Y['cred_vis']['bandw'], colour=COLOUR,
           cache_dir='results/bandsplit_cache', zone=(2, 5))
cm = ChromaticModel(Y['atmo']['heights'], Y['atmo']['Cn2'], science_nm=SCI_NM, zenith_deg=ZENITH)
sp = SplitStudy(st, cm)
ph = Photometry(COLOUR, AREA, cm)
MAGS = (5.0, 6.0, 7.0, 8.0, 9.0, 10.0)                      # magnitude at 900 nm
RMOD_UNMOD, RMOD_MOD, NSUB = 0.0, 2.0, 48

## 1. Anchor: budget vs e2e replays (monochromatic sensors, as in SPECULA)

In [ ]:
E2E = {'single: rMod 0, 900': ('single_wfs900_{t}_r0', lambda mg: [st.sensor(0.0, NSUB, 900., mg)]),
       'single: rMod 2, 900': ('single_wfs900_{t}_r2', lambda mg: [st.sensor(2.0, NSUB, 900., mg)]),
       'fusion: unmod 900 + rMod 2 1300': ('dual_unmod900_{t}', lambda mg: [st.sensor(0.0, NSUB, 900., mg), st.sensor(2.0, NSUB, 1300., mg)]),
       'fusion: unmod 1300 + rMod 2 900': ('dual_unmod1300_{t}', lambda mg: [st.sensor(0.0, NSUB, 1300., mg), st.sensor(2.0, NSUB, 900., mg)])}
rows = []
for chrom in (False, True):
    t = 'chrom' if chrom else 'achrom'
    for name, (case, make) in E2E.items():
        case = case.format(t=t)
        if not os.path.isdir(os.path.join(fr.OUT, case)):       # the achromatic swapped fusion was not simulated
            continue
        rep = fr.replays(case)
        for mg in MAGS + (11.0, 12.0):
            r = st.cached(f'anchor {name} {t} mag{mg:g}', lambda: sp.fused(make(mg), 'min_noise', chromatic=chrom))
            e = rep.get(mg)
            ok = e is not None and e['n_ok'] == e['n']
            rows.append(dict(chromatic=chrom, config=name, mag=mg, budget=st.contrast(r), e2e=e['contrast_vis'] if ok else np.nan,
                             e2e_std=e['contrast_vis_std'] if ok else np.nan))
ANCHOR = pd.DataFrame(rows)
ANCHOR['budget/e2e'] = ANCHOR['budget'] / ANCHOR['e2e']

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True, constrained_layout=True)
for ax, chrom in zip(axs, (False, True)):
    for name, d in ANCHOR[ANCHOR.chromatic == chrom].groupby('config', sort=False):
        ax.plot(d['mag'], d['budget/e2e'], 'o-', label=name)
    ax.axhline(1, color='0.5', lw=1)
    ax.set_title('chromatic' if chrom else 'achromatic', loc='left'); ax.set_xlabel('magnitude (900 nm)')
axs[0].set_ylabel('budget / e2e, contrast 2-5 l/D at 550 nm'); axs[1].legend(fontsize=8)
plt.show()
ANCHOR.pivot_table(index='mag', columns=['chromatic', 'config'], values='budget/e2e').round(2)

## 2. Band model: photons, effective wavelength and chromatic weight of some bands (mag 7)

In [ ]:
BANDS = [(700, 1100), (1100, 1500), (700, 1500), (700, 800), (800, 1500), (700, 900), (900, 1500), (800, 1100)]
ref = ph.band(7.0, 700, 1100)
dn_ref = ref['n_minus_1'] - sp.n_sci
pd.DataFrame([dict(band=f'{lo}-{hi}', flux_vs_700_1100=b['flux'] / ref['flux'], wavelength_nm=b['wavelength_nm'],
                   dn_1e6=(b['n_minus_1'] - sp.n_sci) / 1e-6, chromatic_vs_700_1100=((b['n_minus_1'] - sp.n_sci) / dn_ref) ** 2)
              for lo, hi in BANDS for b in [ph.band(7.0, lo, hi)]]).set_index('band').round(3)

## 3-4. Scan: boundary x role x weights, and the single-sensor baselines

In [ ]:
BOUNDS = np.arange(750.0, 1451.0, 50.0)                     # boundary lb: A = 700-lb, B = lb-1500
WEIGHTS = ('min_noise', 'uniform', 'per_mode')
ROLES = {'unmod blue': True, 'unmod red': False}            # unmodulated sensor on A (blue) or on B (red)

def pair(mg, lb, unmod_blue):
    a, b = ph.sensor(RMOD_UNMOD, NSUB, mg, 700., lb), ph.sensor(RMOD_MOD, NSUB, mg, lb, 1500.)
    return [a, b] if unmod_blue else [dict(a, rMod=RMOD_MOD), dict(b, rMod=RMOD_UNMOD)]

def scan():
    rows = []
    for mg in MAGS:
        for lb in BOUNDS:
            for role, blue in ROLES.items():
                for w in WEIGHTS:
                    r = st.cached(f'split mag{mg:g} lb{lb:g} {role} {w}', lambda: sp.fused(pair(mg, lb, blue), w))
                    rows.append(sp.row(r, mag=mg, boundary=lb, role=role, weights=w))
    return pd.DataFrame(rows)

BASELINES = {'single: unmod 700-1100': lambda mg: [ph.sensor(RMOD_UNMOD, NSUB, mg, 700., 1100.)],
             'single: rMod 2 700-1100': lambda mg: [ph.sensor(RMOD_MOD, NSUB, mg, 700., 1100.)],
             'single: unmod 700-1500': lambda mg: [ph.sensor(RMOD_UNMOD, NSUB, mg, 700., 1500.)],
             'single: rMod 2 700-1500': lambda mg: [ph.sensor(RMOD_MOD, NSUB, mg, 700., 1500.)]}

def baselines():
    return pd.DataFrame([sp.row(st.cached(f'split base {name} mag{mg:g}', lambda: sp.fused(make(mg))), mag=mg, config=name)
                         for name, make in BASELINES.items() for mg in MAGS])

SCAN = st.cached('split scan table', scan, extra=(MAGS, tuple(BOUNDS), WEIGHTS))      # ~15-20 min the first time, then from the cache
BASE = st.cached('split baselines table', baselines, extra=(MAGS,))
TODAY = SCAN[(SCAN.boundary == 1100) & (SCAN.role == 'unmod blue') & (SCAN.weights == 'min_noise')].set_index('mag')['contrast']
SCAN['vs_today'] = SCAN['contrast'] / SCAN['mag'].map(TODAY)
BASE['vs_today'] = BASE['contrast'] / BASE['mag'].map(TODAY)
BASE.pivot(index='mag', columns='config', values='vs_today').round(3)     # baselines relative to today's fusion (< 1: better)

## 5. Results: contrast relative to today's fusion (< 1: better), best split per magnitude

In [ ]:
fig, axs = plt.subplots(len(ROLES), len(WEIGHTS), figsize=(15, 7), sharex=True, sharey=True, constrained_layout=True)
lim = np.nanmax(np.abs(np.log(SCAN['vs_today'])))
norm = TwoSlopeNorm(vcenter=1.0, vmin=np.exp(-lim), vmax=np.exp(lim))
for i, role in enumerate(ROLES):
    for j, w in enumerate(WEIGHTS):
        d = SCAN[(SCAN.role == role) & (SCAN.weights == w)].pivot(index='mag', columns='boundary', values='vs_today')
        im = axs[i, j].pcolormesh(d.columns, d.index, d.values, cmap='RdBu_r', norm=norm, shading='nearest')
        axs[i, j].set_title(f'{role}, {w}', loc='left', fontsize=10)
        axs[i, j].axvline(1100, color='k', lw=0.8, ls=':')
for ax in axs[-1]:
    ax.set_xlabel('boundary [nm] (A = 700-boundary, B = boundary-1500)')
for ax in axs[:, 0]:
    ax.set_ylabel('magnitude (900 nm)')
fig.colorbar(im, ax=axs, label="contrast / today's fusion")
plt.show()

best = SCAN.loc[SCAN.groupby('mag')['contrast'].idxmin()].set_index('mag')
best_single = BASE.loc[BASE.groupby('mag')['contrast'].idxmin()].set_index('mag')
today_e2e = ANCHOR[ANCHOR.chromatic & (ANCHOR.config == 'fusion: unmod 900 + rMod 2 1300')].set_index('mag')['e2e'].reindex(best.index)
TABLE5 = pd.DataFrame({'boundary': best['boundary'], 'role': best['role'], 'weights': best['weights'], 'best / today': best['vs_today'],
              'best / best single': best['contrast'] / best_single['contrast'], 'best single': best_single['config'],
              'today e2e (replays)': today_e2e, 'best, e2e-scaled': best['vs_today'] * today_e2e,
              'fs': best['fs'], 'weight shift (zone)': best['weight_shift_zone'], 'dn zone [1e-6]': best['dn_eff_zone']})
TABLE5.style.format({'best / today': '{:.3f}', 'best / best single': '{:.3f}', 'today e2e (replays)': '{:.3e}', 'best, e2e-scaled': '{:.3e}',
                     'fs': '{:.0f}', 'weight shift (zone)': '{:.2f}', 'dn zone [1e-6]': '{:.2f}', 'boundary': '{:.0f}'})


In [ ]:
fig, (ax, rx) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw=dict(height_ratios=(3, 1.3)), constrained_layout=True)
curves = {"today's fusion (unmod 700-1100 + rMod 2 1100-1500)": TODAY}
for w in WEIGHTS:
    d = SCAN[SCAN.weights == w]
    curves[f'best split, {w}'] = d.loc[d.groupby('mag')['contrast'].idxmin()].set_index('mag')['contrast']
curves['best single (700-1100 or 700-1500)'] = best_single['contrast']
for name, c in curves.items():
    ax.plot(c.index, c.values, 'o-', label=name)
    rx.plot(c.index, c.values / TODAY.loc[c.index].values, 'o-')
ax.set_yscale('log'); ax.set_ylabel('contrast 2-5 l/D at 550 nm (budget)'); ax.legend(fontsize=8)
rx.axhline(1, color='0.5', lw=1); rx.set_ylabel("/ today's fusion"); rx.set_xlabel('magnitude (900 nm)')
plt.show()